DATA CLEANING

Begin Our PySpark Session.

In [107]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("Data Cleaning")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .getOrCreate()
)
# Silence any WARN-level logs from this point forth.
spark.sparkContext.setLogLevel("ERROR")

Define Our Schema:

In [108]:
from pyspark.sql.types import StructType, StructField, StringType

# Here, ensure they are all StringType() as is the original data
raw_schema = StructType([
    StructField("listing_id",          StringType(), True),
    StructField("suburb",              StringType(), True),
    StructField("postcode",            StringType(), True),
    StructField("weekly_rent",         StringType(), True),
    StructField("bond",                StringType(), True),
    StructField("available_date",      StringType(), True),
    StructField("date_listed",         StringType(), True),
    StructField("days_listed",         StringType(), True),
    StructField("bedrooms",            StringType(), True),
    StructField("bathrooms",           StringType(), True),
    StructField("carspaces",           StringType(), True),
    StructField("property_type",       StringType(), True),
    StructField("address",             StringType(), True),
    StructField("lat",                 StringType(), True),
    StructField("lon",                 StringType(), True),
    StructField("scraped_date",        StringType(), True),
    StructField("domain_page_id",      StringType(), True),
    StructField("property_id",         StringType(), True),
    StructField("photo_count",         StringType(), True),
    StructField("video_count",         StringType(), True),
    StructField("floorplans_count",    StringType(), True),
    StructField("virtual_tour",        StringType(), True),
    StructField("primary_type",        StringType(), True),
    StructField("secondary_type",      StringType(), True),
    StructField("agency",              StringType(), True),
    StructField("agency_id",           StringType(), True),
    StructField("agent_names",         StringType(), True),
    StructField("structured_features", StringType(), True),
    StructField("url",                 StringType(), True),
    StructField("land_area",           StringType(), True),
])


Read in our data using our schema. 
Note: We must allow multi line csv reading since there are multiple rows within our data where the "structured_features" column contains multiline descriptions. If we do not clarify this, our data will not be alligned properly when read in.

In [109]:
path = "../rea_data/landing/domain/Data/vic_rentals_all.csv"
df_landing = spark.read.csv(
    path,
    schema=raw_schema,
    header=True,
    quote='"',
    escape='"',
    multiLine=True
)
df_landing.printSchema()

root
 |-- listing_id: string (nullable = true)
 |-- suburb: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- weekly_rent: string (nullable = true)
 |-- bond: string (nullable = true)
 |-- available_date: string (nullable = true)
 |-- date_listed: string (nullable = true)
 |-- days_listed: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- carspaces: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- address: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- scraped_date: string (nullable = true)
 |-- domain_page_id: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- photo_count: string (nullable = true)
 |-- video_count: string (nullable = true)
 |-- floorplans_count: string (nullable = true)
 |-- virtual_tour: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- secondary_type: string (null

Lets define a function to check null values within our data for use throughout the preprocessing procedure.

In [110]:
from functools import reduce

def null_check(df):
    has_null_condition = reduce(
        lambda acc, c: acc | F.col(c).isNull(),
        df.columns,
        F.lit(False)
    )

    # Count total rows, rows with atleast one null and count how many nulls occur per
    # data field.
    summary = df.agg(
        F.count(F.lit(1)).alias("total_rows"),
        F.sum(has_null_condition.cast("int")).alias("rows_with_at_least_one_null"),
        *[F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}") for c in df.columns]
    ).collect()[0]
    
    total = summary["total_rows"]
    rows_with_at_least_one_null = summary["rows_with_at_least_one_null"]
    
    print(f"Total rows: {total:,}")
    print(f"Rows with at least one null value: {rows_with_at_least_one_null:,}")
    print(f"Percentage of rows with atleast one NULL: {round(100 * rows_with_at_least_one_null / total, 2)}%")
    print()
    for c in df.columns:
        print(f"{c:25s} {summary[f'null_{c}']:>12,}")


In [111]:
null_check(df_landing)

Total rows: 12,717
Rows with at least one null value: 12,717
Percentage of rows with atleast one NULL: 100.0%

listing_id                           0
suburb                               0
postcode                             0
weekly_rent                        279
bond                               782
available_date                     136
date_listed                          4
days_listed                          4
bedrooms                           125
bathrooms                           51
carspaces                        1,803
property_type                        0
address                            104
lat                                  4
lon                                  4
scraped_date                         0
domain_page_id                       1
property_id                     10,145
photo_count                          4
video_count                          4
floorplans_count                     4
virtual_tour                         4
primary_type                   

Now, lets explicitly type cast so we can tell when data was genuinely blank in the source rather than a silent parsing error resulting in missing data. Note: We will only do this for relevant columns and leave the rest as strings for now (They will be removed later).

In [112]:
df_clean = (
    df_landing
    # --- Numeric IDs: source stores these as floats ("21230.0"), 
    #     cast via double first, then to long. Use try_cast defensively. ---
    .withColumn("listing_id",     F.expr("try_cast(listing_id AS DOUBLE)").cast("long"))
    .withColumn("agency_id",      F.expr("try_cast(agency_id AS DOUBLE)").cast("long"))

    # property_id: alphanumeric code (e.g. "SD-2640-DB") — NOT numeric, leave as string
    # no cast applied — stays StringType from raw_schema

    # --- Money: clean decimal strings ("460.0", "1994.0") — direct cast is safe ---
    .withColumn("weekly_rent", F.expr("try_cast(weekly_rent AS DOUBLE)"))
    .withColumn("bond",        F.expr("try_cast(bond AS DOUBLE)"))

    # --- Dates ---
    .withColumn("date_listed",  F.to_date("date_listed", "yyyy-MM-dd"))
    .withColumn("scraped_date", F.to_timestamp("scraped_date", "yyyy-MM-dd HH:mm:ss"))
    # available_date left as string, per earlier decision

    # --- Simple numerics ---
    .withColumn("days_listed",      F.expr("try_cast(days_listed AS DOUBLE)"))
    .withColumn("bedrooms",         F.expr("try_cast(bedrooms AS DOUBLE)"))
    .withColumn("bathrooms",        F.expr("try_cast(bathrooms AS DOUBLE)"))
    .withColumn("carspaces",        F.expr("try_cast(carspaces AS DOUBLE)"))
    .withColumn("photo_count",      F.expr("try_cast(photo_count AS DOUBLE)").cast("integer"))
    .withColumn("video_count",      F.expr("try_cast(video_count AS DOUBLE)").cast("integer"))
    .withColumn("floorplans_count", F.expr("try_cast(floorplans_count AS DOUBLE)").cast("integer"))
    .withColumn("lat", F.expr("try_cast(lat AS DOUBLE)"))
    .withColumn("lon", F.expr("try_cast(lon AS DOUBLE)"))

    # --- Boolean ---
    .withColumn("virtual_tour", F.col("virtual_tour").cast("boolean"))

    # --- land_area: extract numeric part ---
    .withColumn("land_area_sqm",
        F.regexp_extract(F.col("land_area"), r"(\d+\.?\d*)", 1).cast("double"))
)

In [113]:
df_clean.printSchema()

root
 |-- listing_id: long (nullable = true)
 |-- suburb: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- weekly_rent: double (nullable = true)
 |-- bond: double (nullable = true)
 |-- available_date: string (nullable = true)
 |-- date_listed: date (nullable = true)
 |-- days_listed: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- carspaces: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- address: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- scraped_date: timestamp (nullable = true)
 |-- domain_page_id: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- photo_count: integer (nullable = true)
 |-- video_count: integer (nullable = true)
 |-- floorplans_count: integer (nullable = true)
 |-- virtual_tour: boolean (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- secondary_type: string (n

Once again, lets do a null audit to ensure we haven't accidently removed any missing data.

In [114]:
null_check(df_clean)

Total rows: 12,717
Rows with at least one null value: 12,717
Percentage of rows with atleast one NULL: 100.0%

listing_id                           0
suburb                               0
postcode                             0
weekly_rent                        279
bond                               782
available_date                     136
date_listed                          4
days_listed                          4
bedrooms                           125
bathrooms                           51
carspaces                        1,803
property_type                        0
address                            104
lat                                  4
lon                                  4
scraped_date                         0
domain_page_id                       1
property_id                     10,145
photo_count                          4
video_count                          4
floorplans_count                     4
virtual_tour                         4
primary_type                   

Lets also clean up our "structued_features" descriptions so they are all comma seperated properly and consistent with other rows.

In [115]:
row = df_clean.filter(F.col("listing_id") == "17747859")
row.show(truncate=False, vertical=True)

-RECORD 0--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 listing_id          | 17747859                                                                                                                                                                                                                                                                                                                                  
 suburb              | DOCKLANDS                                                                                                                                                                                                                                                                    

In [116]:
df_clean = df_clean.withColumn(
    "structured_features",
    F.regexp_replace(
        F.col("structured_features"),
        r"[\r\n]+-\s*",   # matches \r\n or \n followed by "- " (the bullet style)
        ", "              # replace with a plain comma separator
    )
)

In [117]:
row = df_clean.filter(F.col("listing_id") == "17747859")
row.show(truncate=False, vertical=True)

-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 listing_id          | 17747859                                                                                                                                                                                                                                                                                                                  
 suburb              | DOCKLANDS                                                                                                                                                                                                                                                                                                    

We used a specific case ("listing_id=17747859") to illustrate our point, but must check that it has been fixed for every such row.

In [118]:
remaining_bullets = df_clean.filter(
    F.col("structured_features").rlike(r"[\r\n]-\s")
)
remaining_bullets.count()

0

### Dealing with structured_features

In [119]:
from pyspark.ml.feature import CountVectorizer

# --- 1. Strip section headers (e.g. "BUILDING AMENITIES") before splitting ---
header_pattern = r"(BUILDING AMENITIES|INDOOR FEATURES|OUTDOOR FEATURES|CLIMATE CONTROL)\s*,?\s*"
df_clean = df_clean.withColumn(
    "structured_features_clean",
    F.regexp_replace(F.col("structured_features"), header_pattern, "")
)

# --- 2. Split into array, trim whitespace, lowercase each element ---
df_clean = df_clean.withColumn(
    "features_array",
    F.expr("""
        transform(
            split(trim(structured_features_clean), ','),
            x -> lower(trim(x))
        )
    """)
).withColumn(
    "features_array",
    F.expr("filter(features_array, x -> x != '')")  # drop empty strings from trailing commas
)

# --- Explode features_array into one row per (listing, feature), then count distinct values ---
feature_counts = (
    df_clean
    .select(F.explode("features_array").alias("feature"))
    .groupBy("feature")
    .count()
    .orderBy(F.desc("count"))
)

total_unique = feature_counts.count()
print(f"Total unique feature strings: {total_unique}")

feature_counts.show(total_unique, truncate=False)

Total unique feature strings: 532
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|feature                                                                                                                                                                                                                                |count|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|built in wardrobes                                                                                                                                                                                                                     |6690 |
|heati

In [120]:
# --- Junk filter: flag likely listing-description leakage vs genuine short features ---

feature_counts_flagged = (
    feature_counts
    .withColumn("word_count", F.size(F.split(F.col("feature"), r"\s+")))
    .withColumn("char_length", F.length(F.col("feature")))
    .withColumn(
        "is_likely_junk",
        (F.col("word_count") > 6) |                                  # long free-text phrases
        (F.col("char_length") > 45) |                                 # long strings generally
        F.col("feature").rlike(r"\.\s|,\s.*,\s") |                    # embedded sentence punctuation
        F.col("feature").rlike(r"\b(will be|please|provided|included at|works during|suitable for)\b")
    )
)

# --- Review what gets flagged before deleting anything ---
print("=== Flagged as likely junk (review before dropping) ===")
feature_counts_flagged.filter("is_likely_junk").orderBy(F.desc("count")).show(100, truncate=False)

print("=== Kept as likely genuine features ===")
feature_counts_flagged.filter("NOT is_likely_junk").orderBy(F.desc("count")).show(100, truncate=False)

n_junk = feature_counts_flagged.filter("is_likely_junk").count()
n_kept = feature_counts_flagged.filter("NOT is_likely_junk").count()
print(f"\nFlagged as junk: {n_junk} / {n_junk + n_kept} unique features")

=== Flagged as likely junk (review before dropping) ===
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------+-----------+--------------+
|feature                                                                                                                                                                                                                                |count|word_count|char_length|is_likely_junk|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------+-----------+--------------+
|energy eff. rating: 6                                                                                        

In [121]:
# --- Extract "energy eff. rating: X" into a proper numeric column ---

df_clean = df_clean.withColumn(
    "energy_eff_rating",
    F.expr("""
        aggregate(
            filter(features_array, x -> x rlike 'energy eff\\\\. rating: [0-9.]+'),
            cast(null as double),
            (acc, x) -> try_cast(regexp_extract(x, 'energy eff\\\\. rating: ([0-9.]+)', 1) as double)
        )
    """)
)

# --- Remove the raw "energy eff. rating: X" strings from features_array now that they're captured ---
df_clean = df_clean.withColumn(
    "features_array",
    F.expr("filter(features_array, x -> NOT (x rlike 'energy eff\\\\. rating: [0-9.]+'))")
)

# --- Sanity check ---
df_clean.select("listing_id", "energy_eff_rating").filter(F.col("energy_eff_rating").isNotNull()).show(15)
df_clean.select("energy_eff_rating").summary("count", "min", "max", "mean").show()

+----------+-----------------+
|listing_id|energy_eff_rating|
+----------+-----------------+
|  17703824|              6.0|
|  17391686|              6.0|
|  17743107|              7.0|
|  17561797|              5.0|
|  17722774|              8.5|
|  17493594|              6.0|
|  17513777|              6.0|
|  17514065|              6.0|
|  17571887|              6.0|
|  17666119|              6.0|
|  17714175|              6.0|
|  17733347|              6.0|
|  17649973|              5.0|
|  17741703|              7.0|
|  17725067|              7.0|
+----------+-----------------+
only showing top 15 rows
+-------+-----------------+
|summary|energy_eff_rating|
+-------+-----------------+
|  count|               51|
|    min|              0.5|
|    max|              8.6|
|   mean| 5.74313725490196|
+-------+-----------------+



In [122]:
# Collect feature counts to pandas (only 532 rows — trivial size)
pdf_counts = feature_counts.toPandas()

# pip install rapidfuzz --break-system-packages   # if not already installed
from rapidfuzz import fuzz, process

# --- Exclude features with digits or "un-" prefixes from fuzzy clustering ---
# These need separate handling: digits -> numeric columns, un-prefixed -> opposite category, not a fuzzy merge
exclude_mask = pdf_counts["feature"].str.contains(r"\d", regex=True) | pdf_counts["feature"].str.match(r"^un\w+")
excluded_df = pdf_counts[exclude_mask]
fuzzy_df = pdf_counts[~exclude_mask]

print(f"Excluded from fuzzy clustering: {len(excluded_df)} (review separately)")
excluded_df.sort_values("count", ascending=False)

/home/jovinguyen/miniconda3/envs/tf_env/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Excluded from fuzzy clustering: 52 (review separately)


,feature,count
35,pool/spa count: 0,253
120,energy eff. rating: 6,20
154,garage spaces: 1,10
179,energy eff. rating: 5,7
183,energy eff. rating: 7,6
184,unfurnished,6
185,car spaces: 1,6
215,garage spaces: 2,4
219,energy eff. rating: 6.5,4
227,carport spaces: 1,4


In [123]:
features = fuzzy_df["feature"].tolist()
counts = dict(zip(fuzzy_df["feature"], fuzzy_df["count"]))

clusters = {}
assigned = set()

# Process from most-frequent to least, so common terms become canonical cluster labels
for feat in sorted(features, key=lambda f: -counts[f]):
    if feat in assigned:
        continue
    matches = process.extract(
        feat, features, scorer=fuzz.token_sort_ratio, limit=10, score_cutoff=87
    )
    group = [m[0] for m in matches if m[0] not in assigned]
    if len(group) > 1:
        clusters[feat] = group          # feat = canonical name (most frequent form)
        assigned.update(group)

# Show candidate clusters for manual review
for canonical, group in clusters.items():
    if len(group) > 1:
        print(f"{canonical!r}  <-  {group}")

'internal laundry'  <-  ['internal laundry', 'laundry: internal']
'balcony / deck'  <-  ['balcony / deck', 'deck / balcony']
'remote garage'  <-  ['remote garage', 'garage remote']
'ducted heating'  <-  ['ducted heating', 'gas ducted heating']
'split system heating'  <-  ['split system heating', 'split system (heating)', 'split-system heating']
'outdoor entertainment area'  <-  ['outdoor entertainment area', 'outdoor entertaining area', 'outdoor entertainment']
'courtyard'  <-  ['courtyard', 'coutyard']
'split system air con'  <-  ['split system air con', 'split system aircon', 'split system (air con)', 'split-system air con']
'split system air conditioning'  <-  ['split system air conditioning', 'split system airconditioning', 'split-system air conditioning', 'split-system air-conditioning']
'gas heating'  <-  ['gas heating', 'heating: gas']
'cooking: gas hotplates and electric oven'  <-  ['cooking: gas hotplates and electric oven', 'cooking: electric hotplates & gas oven', 'cooking: 

In [124]:
merge_map = {
    # --- Formatting/spelling variants -> canonical name ---
    "laundry: internal": "internal laundry",
    "deck / balcony": "balcony / deck",
    "garage remote": "remote garage",
    "gas ducted heating": "ducted heating",
    "split system (heating)": "split system heating",
    "split-system heating": "split system heating",
    "outdoor entertaining area": "outdoor entertainment area",
    "outdoor entertainment": "outdoor entertainment area",
    "coutyard": "courtyard",

    # split system air con group (merged together — "air con" == "air conditioning")
    "split system aircon": "split system air conditioning",
    "split system (air con)": "split system air conditioning",
    "split-system air con": "split system air conditioning",
    "split system air con": "split system air conditioning",
    "split system airconditioning": "split system air conditioning",
    "split-system air conditioning": "split system air conditioning",
    "split-system air-conditioning": "split system air conditioning",
    "split system ac": "split system air conditioning",
    "split system": "split system air conditioning",

    "heating: gas": "gas heating",
    "cooking: electric hotplates & gas oven": "cooking: gas hotplates and electric oven",
    "cooking: electric hotplates & oven": "cooking: gas hotplates and electric oven",
    "cooking: gas oven and hotplates": "cooking: gas hotplates and electric oven",
    "reverse cycle aircon": "reverse cycle air con",
    "ducted air-conditioning": "ducted air conditioning",
    "tennis courts": "tennis court",
    "spa (inside)": "inside spa",
    "modern kitchen": "kitchen: modern",
    "basement car parking": "car parking - basement",
    "living areas": "living area",
    "hot water service": "gas hot water service",  # NOTE: see caveat below — fuel type unconfirmed
    "kitchen: renovated": "renovated kitchen",
    "laundry: separate": "separate laundry",

    # in-ground pool group (kept SEPARATE from above-ground — see correction below)
    "pool (inground)": "inground pool",
    "pool (in ground)": "inground pool",

    # above-ground pool group (corrected: no longer merged with "below ground")
    "pool (above ground)": "above ground pool",
    "above-ground pool": "above ground pool",

    "heating electric": "electric heating",
    "heating: electric": "electric heating",
    "split system heating / cooling": "split system heating and cooling",
    "split system heating & cooling": "split system heating and cooling",
    "hyrdonic heating": "hydronic heating",
    "heating other": "heating: other",
    "carports": "carport",
    "electrical cooktop": "electric cooktop",
    "backyard access": "access to backyard",

    # euro laundry group (merged together — these are all the same category)
    "euro laundry": "laundry: euro",
    "laundry: european laundry": "laundry: euro",
    "laundry: european": "laundry: euro",
    "european laundry": "laundry: euro",

    "reverse cycle ac": "reverse cycle",
    "integrated fridge/freezer": "intergrated frige/freezer",  # NOTE: canonical form has a typo — see below
}

In [125]:
from itertools import chain

# Build a Spark map literal from the Python merge_map dict
mapping_col = F.create_map([F.lit(x) for pair in merge_map.items() for x in pair])

# Apply it: for each element in features_array, look it up in the map;
# if not found (not in merge_map), keep the original value via coalesce
df_clean = df_clean.withColumn(
    "features_array",
    F.transform("features_array", lambda x: F.coalesce(mapping_col[x], x))
)

In [126]:
feature_counts_after = (
    df_clean
    .select(F.explode("features_array").alias("feature"))
    .groupBy("feature")
    .count()
    .orderBy(F.desc("count"))
)
print(f"Total unique feature strings after merge: {feature_counts_after.count()}")
feature_counts_after.show(50, truncate=False)

Total unique feature strings after merge: 466
+-----------------------------+-----+
|feature                      |count|
+-----------------------------+-----+
|built in wardrobes           |6690 |
|heating                      |5710 |
|dishwasher                   |5456 |
|secure parking               |4966 |
|internal laundry             |3997 |
|air conditioning             |3796 |
|balcony / deck               |2510 |
|gas                          |2421 |
|floorboards                  |1904 |
|bath                         |1881 |
|intercom                     |1629 |
|ensuite                      |1621 |
|fully fenced                 |1433 |
|study                        |1280 |
|furnished                    |1128 |
|split system air conditioning|1089 |
|pets allowed                 |1047 |
|close to shops               |965  |
|split system heating         |948  |
|close to transport           |947  |
|close to schools             |932  |
|remote garage                |919  |
|she

In [127]:
from pyspark.sql.types import ArrayType, StringType

df_clean = df_clean.withColumn(
    "features_array",
    F.coalesce(F.col("features_array"), F.array().cast(ArrayType(StringType())))
)

# Confirm no nulls remain before fitting
null_count = df_clean.filter(F.col("features_array").isNull()).count()
print(f"Remaining nulls in features_array: {null_count}")

Remaining nulls in features_array: 0


In [128]:
from pyspark.ml.feature import CountVectorizer

cv = CountVectorizer(
    inputCol="features_array",
    outputCol="features_vector",
    binary=True,
    minDF=100
)
cv_model = cv.fit(df_clean)
df_clean = cv_model.transform(df_clean)

print(f"Vocabulary size after minDF filter: {len(cv_model.vocabulary)}")
print(cv_model.vocabulary)

Vocabulary size after minDF filter: 52
['built in wardrobes', 'heating', 'dishwasher', 'secure parking', 'internal laundry', 'air conditioning', 'balcony / deck', 'gas', 'floorboards', 'bath', 'intercom', 'ensuite', 'fully fenced', 'study', 'furnished', 'split system air conditioning', 'pets allowed', 'close to shops', 'split system heating', 'close to transport', 'close to schools', 'remote garage', 'shed', 'ducted heating', 'garden / courtyard', 'gym', 'outdoor entertainment area', 'balcony', 'broadband internet access', 'alarm system', 'swimming pool', 'courtyard', 'solar panels', 'fireplace(s)', 'pool/spa count: 0', 'bathtub', 'ensuite(s)', 'carpeted', 'gas appliances', 'evaporative cooling', 'openable windows', 'broadband', 'rumpus room', 'ducted cooling', 'broadband internet available', 'city views', 'gas heating', 'deck', 'laundry', 'air-conditioner', 'separate dining room', 'double glazed windows']


### Save final CSV into a parquet file

~Jovi: We should save it as a parquet file so that metadata/schema is stored by defualt. In addition, running the cell below just updates vic_rentals_clean.parquet if any changes are made down the road.

In [129]:
# Save the cleaned DataFrame to Parquet format
output_path = "../rea_data/curated/vic_rentals_clean.parquet"

df_clean.write.mode("overwrite").parquet(output_path)

# Verify: reload without specifying a schema and confirm it matches df_clean exactly
df_verify = spark.read.parquet(output_path)
df_verify.printSchema()

assert df_verify.schema == df_clean.schema, "Schema mismatch after round-trip!"
print(f"Saved {df_verify.count():,} rows to {output_path} with schema intact.")

root
 |-- listing_id: long (nullable = true)
 |-- suburb: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- weekly_rent: double (nullable = true)
 |-- bond: double (nullable = true)
 |-- available_date: string (nullable = true)
 |-- date_listed: date (nullable = true)
 |-- days_listed: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- carspaces: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- address: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- scraped_date: timestamp (nullable = true)
 |-- domain_page_id: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- photo_count: integer (nullable = true)
 |-- video_count: integer (nullable = true)
 |-- floorplans_count: integer (nullable = true)
 |-- virtual_tour: boolean (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- secondary_type: string (n

AssertionError: Schema mismatch after round-trip!

In [ ]:
# NOTE FOR LATER: IF WE TURN STRUCTURED FEATURES INTO COMMA SEPERATED ARRAYS OF FEATURES, something like "BUILDING AMENITIES" 
# IS NOT A FEATURE SO MUST BE PREPROCESSED BEFORE WE DO A STEP LIKE THAT.